SHA-256 fingerprint Calculation

In [0]:

from pathlib import Path
import hashlib

# Original January 2025 source file
source_path = Path(
    "/Volumes/workspace/default/raw_landing/yellow_tripdata_2025-01.parquet"
)

if not source_path.is_file():
    raise FileNotFoundError(source_path)

# Calculate the file fingerprint in chunks
sha256 = hashlib.sha256()

with source_path.open("rb") as file:
    for chunk in iter(
        lambda: file.read(1024 * 1024),
        b""
    ):
        sha256.update(chunk)

file_hash = sha256.hexdigest()
file_bytes = source_path.stat().st_size

print("Source month: 2025-01")
print("File:", source_path.name)
print("File size:", file_bytes)
print("SHA-256:", file_hash)

assert len(file_hash) == 64
assert file_bytes == 59158238

print("PASS: Source fingerprint calculated")


Manifest Delta table

In [0]:

# Create a dedicated control schema
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.control
""")

# Create the source tracking table
spark.sql("""
    CREATE TABLE IF NOT EXISTS
        workspace.control.source_file_manifest (
            source_type STRING,
            source_month STRING,
            source_path STRING,
            content_sha256 STRING,
            file_bytes BIGINT,
            first_seen_at TIMESTAMP,
            processing_status STRING,
            published_at TIMESTAMP
        )
    USING DELTA
""")

print("PASS: Source manifest table exists")


Source file register

In [0]:

# Prepare one source-version record
candidate = spark.createDataFrame(
    [(
        "yellow_taxi",
        "2025-01",
        str(source_path),
        file_hash,
        file_bytes
    )],
    """
    source_type STRING,
    source_month STRING,
    source_path STRING,
    content_sha256 STRING,
    file_bytes LONG
    """
)

candidate.createOrReplaceTempView("source_candidate")

# Insert this file version only if it doesn't exist
spark.sql("""
    MERGE INTO workspace.control.source_file_manifest AS target
    USING source_candidate AS src

    ON target.source_type = src.source_type
       AND target.source_month = src.source_month
       AND target.content_sha256 = src.content_sha256

    WHEN NOT MATCHED THEN INSERT (
        source_type,
        source_month,
        source_path,
        content_sha256,
        file_bytes,
        first_seen_at,
        processing_status,
        published_at
    )
    VALUES (
        src.source_type,
        src.source_month,
        src.source_path,
        src.content_sha256,
        src.file_bytes,
        current_timestamp(),
        'OBSERVED',
        NULL
    )
""")

# Inspect the result
display(
    spark.table("workspace.control.source_file_manifest")
    .filter("source_month = '2025-01'")
)


Replay trascking Verification

In [0]:

manifest = spark.table(
    "workspace.control.source_file_manifest"
)

matching = manifest.filter(
    (manifest.source_type == "yellow_taxi") &
    (manifest.source_month == "2025-01") &
    (manifest.content_sha256 == file_hash)
)

assert matching.count() == 1

print("PASS: Same source version registered only once")
